# Comparación de modelos de sueño — Jhoan Saavedra

Notebook sencillo para comparar Random Forest y LightGBM usando exactamente el mismo preprocesamiento reutilizable que consumirá FastAPI. La construcción del dataset, división por sujeto, métricas y tracking viven en el paquete `sleep-staging`.

## 1. Configuración

Para una prueba rápida puede asignarse `MAX_RECORDS = 4`. Para el experimento completo debe permanecer en `None`. MLflow está desactivado por defecto.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / 'data.dvc').is_file())
PACKAGE_SRC = REPO_ROOT / 'packages' / 'sleep-staging' / 'src'
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

from IPython.display import display
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import discover_sleep_telemetry_records
from sleep_staging.training import (
    MlflowConfig, build_supervised_dataset, compare_evaluations,
    create_lightgbm_classifier, create_random_forest_classifier,
    display_evaluation, split_by_subject, train_and_evaluate,
)

DATA_DIR = REPO_ROOT / 'data' / 'sleep-telemetry'
SEED = 42
VALIDATION_SIZE = 0.20
MAX_RECORDS = None
MLFLOW = MlflowConfig(
    enabled=False,
    experiment_name='sleep_staging_model_comparison',
    tracking_uri='sqlite:///mlflow.db',
)
PREPROCESSING = PreprocessingConfig()
PIPELINE = PreprocessingPipeline(PREPROCESSING)

## 2. Preprocesamiento y dataset

`build_supervised_dataset` ejecuta `PreprocessingPipeline.transform_edf` para cada PSG. El hipnograma se incorpora después y solo aporta las etiquetas de entrenamiento.

In [ ]:
records = discover_sleep_telemetry_records(DATA_DIR)
records = records if MAX_RECORDS is None else records[:MAX_RECORDS]
dataset = build_supervised_dataset(records, PIPELINE)

print(f'Dataset: {dataset.features.shape[0]} épocas x {dataset.features.shape[1]} features')
print(f'Sujetos: {dataset.metadata.subject_id.nunique()} | Registros: {dataset.metadata.record_id.nunique()}')
display(dataset.labels.value_counts().rename('épocas').sort_index().to_frame())

## 3. División entrenamiento/validación por sujeto

`VALIDATION_SIZE` controla la proporción aproximada de sujetos reservada. Las dos noches de una persona siempre quedan en el mismo conjunto.

In [ ]:
split = split_by_subject(dataset, validation_size=VALIDATION_SIZE, random_state=SEED)
print(f'Train: {split.X_train.shape} | {len(split.train_subjects)} sujetos')
print(f'Validación: {split.X_validation.shape} | {len(split.validation_subjects)} sujetos')
print(f'Sujetos de validación: {split.validation_subjects}')

## 4. Modelo Random Forest

In [ ]:
RF_PARAMS = {'n_estimators': 500, 'min_samples_leaf': 2}
random_forest = create_random_forest_classifier(random_state=SEED, **RF_PARAMS)
rf_evaluation = train_and_evaluate(
    random_forest, split, model_name='Random Forest',
    parameters=RF_PARAMS, mlflow_config=MLFLOW,
)

## 5. Modelo LightGBM

LightGBM es una elección explícita. Si la dependencia no está instalada, esta celda genera un error con la instrucción de instalación; nunca cambia silenciosamente a Random Forest.

In [ ]:
LGBM_PARAMS = {'n_estimators': 700, 'learning_rate': 0.05, 'num_leaves': 63}
lightgbm = create_lightgbm_classifier(random_state=SEED, **LGBM_PARAMS)
lgbm_evaluation = train_and_evaluate(
    lightgbm, split, model_name='LightGBM',
    parameters=LGBM_PARAMS, mlflow_config=MLFLOW,
)

## 6. Resultados y matrices de confusión

In [ ]:
display(compare_evaluations(rf_evaluation, lgbm_evaluation).round(4))
display_evaluation(rf_evaluation)
display_evaluation(lgbm_evaluation)